Here We Check How Contributing Factors Are Allocated

In [ ]:
import pandas as pd
import numpy as np
from pprint import pprint

# Merge Raw Datasets

In [27]:
from pathlib import Path

li = [
    pd.read_csv(f, index_col=0, header=[0, 1], low_memory=False)
    for f in Path("../../raw").glob("*.csv")
    if "merged" not in f.name
]

# Combine all DataFrames
raw_df = pd.concat(li, axis=0)

# Remove 'unnamed' column at end (because the CSVs contain ',\n' by default)
raw_df = raw_df.iloc[:, :-1]

# Remove those with duplicate ACNs
unique = raw_df[~raw_df.index.duplicated(keep="first")]

In [69]:
flat = unique.copy()
flat.columns = [f"{a.replace(' ', "_")}_{b.replace(' ', "_")}" for a, b in flat.columns]
flat = flat.rename({"Assessments_Contributing_Factors_/_Situations" : "contributing_factors", "Assessments_Primary_Problem" : "primary_problem"}, axis=1)
flat.index.name = 'acn'
flat.head()

,Time_Date,Time_Local_Time_Of_Day,Place_Locale_Reference,Place_State_Reference,Place_Relative_Position.Angle.Radial,Place_Relative_Position.Distance.Nautical_Miles,Place_Altitude.AGL.Single_Value,Place_Altitude.MSL.Single_Value,Place_Latitude_/_Longitude_(UAS),Environment_Flight_Conditions,...,Events_Detector,Events_When_Detected,Events_Result,contributing_factors,primary_problem,Report_1_Narrative,Report_1_Callback,Report_2_Narrative,Report_2_Callback,Report_1_Synopsis
acn,,,,,,,,,,,,,,,,,,,,,
2260174,202507,1201-1800,BWI.Airport,MD,NaN,NaN,NaN,900.0,NaN,VMC,...,Person Air Traffic Control,In-flight,General None Reported / Taken,Procedure; Human Factors; ATC Equipment / Nav ...,Ambiguous,Aircraft vectored in within 1NM to final appro...,NaN,NaN,NaN,Air carrier Captain reported a low altitude al...
2260249,202507,0601-1200,ZZZ.Airport,US,NaN,NaN,NaN,NaN,NaN,NaN,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Executed Go Around / Missed Approa...,Human Factors; Aircraft,Ambiguous,While on short final we received a glideslope ...,NaN,NaN,NaN,Air carrier pilot reported receiving an aircra...
2260370,202507,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,35000.0,NaN,VMC,...,Person Flight Crew,In-flight,Air Traffic Control Issued New Clearance; Air ...,Aircraft,Aircraft,Flying at cruise; FL350; the FO was the PF and...,NaN,At cruise; FL350; during level-off; the Captai...,NaN,B737 flight crew reported observing a slowing ...
2261277,202507,1801-2400,ZZZ.Airport,US,NaN,2.6,160.0,NaN,NaN,NaN,...,Person Flight Crew,In-flight,General None Reported / Taken,Human Factors,Human Factors,On Day 0 around XA:30; I forgot to get LAANC a...,Reporter stated TRUST certificate was obtained...,NaN,NaN,Recreational / Hobbyist UAS pilot reported fly...
2261317,202507,NaN,ZZZ.Airport,US,NaN,NaN,NaN,9000.0,NaN,VMC,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Became Reoriented; Flight Crew Reg...,Weather; Human Factors; Procedure,Procedure,Divert into ZZZ from ZZZ1. FO flying. Vectors ...,NaN,Extremely strong winds blew us off the LOC whi...,NaN,B737 flight crew reported the pilot flying in ...


# Analyze Contributing Factors

In [70]:
df = flat.copy()
cf = df.contributing_factors.map(lambda x : x.replace(' ', '').split(";") if isinstance(x, str) else ["None"])
cf

acn
2260174    [Procedure, HumanFactors, ATCEquipment/NavFaci...
2260249                             [HumanFactors, Aircraft]
2260370                                           [Aircraft]
2261277                                       [HumanFactors]
2261317                   [Weather, HumanFactors, Procedure]
                                 ...                        
1780435                              [Weather, HumanFactors]
1782430                                   [Airport, Weather]
1791911                                            [Weather]
1883818                                            [Weather]
2012024                              [HumanFactors, Weather]
Name: contributing_factors, Length: 38161, dtype: object

In [71]:
cf.loc[2260174]

['Procedure', 'HumanFactors', 'ATCEquipment/NavFacility/Buildings']

In [72]:
from sklearn.preprocessing import MultiLabelBinarizer

MLB = MultiLabelBinarizer()
onehot = pd.DataFrame(MLB.fit_transform(cf), columns= MLB.classes_, index=cf.index)
onehot

,ATCEquipment/NavFacility/Buildings,Aircraft,Airport,AirspaceStructure,ChartOrPublication,CompanyPolicy,Environment-NonWeatherRelated,Equipment/Tooling,HumanFactors,Incorrect/NotInstalled/UnavailablePart,LogbookEntry,MEL,Manuals,None,Procedure,SoftwareandAutomation,Staffing,Weather
acn,,,,,,,,,,,,,,,,,,
2260174,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
2260249,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2260370,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2261277,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2261317,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1780435,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
1782430,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1791911,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [74]:
onehot.to_csv("all_contributing_factors.csv")

In [60]:
onehot[['Procedure', 'HumanFactors']].value_counts(normalize=True)

Procedure  HumanFactors
0          1               0.321401
1          1               0.280889
0          0               0.280601
1          0               0.117109
Name: proportion, dtype: float64

In [65]:
cont_hf_proc = onehot.join(df)[['primary_problem', 'Procedure', 'HumanFactors']]
hfp = set(['Human Factors', 'Procedure'])
cont_hf_proc = cont_hf_proc[cont_hf_proc.primary_problem.isin(hfp)]
cont_hf_proc.value_counts(normalize=True)

primary_problem  Procedure  HumanFactors
Human Factors    0          1               0.438512
Procedure        1          1               0.262143
Human Factors    1          1               0.157825
Procedure        1          0               0.141520
Name: proportion, dtype: float64